# aqualens quickstart

Monitor any water body from Sentinel-2 imagery: pick coordinates and dates, run one function, get a drawdown analysis with uncertainty.

You need a free [Copernicus Data Space](https://dataspace.copernicus.eu/) account — the auth cell below prints a login link.

Downloads and results are kept in Google Drive under `MyDrive/aqualens/`, so interrupted runs resume where they left off.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet --upgrade git+https://github.com/jxbyr/aqualens.git

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from aqualens import AOI, PipelineConfig, run_pipeline
from aqualens.acquisition import connect

connection = connect()  # prints a login URL on first run

## Configure

The default below is the southern Aral Sea — the site of the original GEOL0069 project — sampled every August. Change `lat`, `lon`, `buffer_km`, and the dates to monitor any other water body.

Each epoch runs as a server-side batch job on the Copernicus Data Space (expect a few minutes per epoch on the first run; cached afterwards). For a quick smoke test, start with 2–3 dates before running a long series.

Tip: to persist downloads and results across Colab sessions, mount Google Drive and point `out_dir` / `cache_dir` there.

In [ ]:
config = PipelineConfig(
    aoi=AOI(lat=45.0, lon=59.0, buffer_km=60),
    dates=PipelineConfig.date_range("2018-08-15", "2025-08-15", "yearly"),
    window_days=45,
    max_cloud_cover=40,
    resolution_m=60,
    cache_dir="/content/drive/MyDrive/aqualens/aqualens_cache",
    out_dir="/content/drive/MyDrive/aqualens/aqualens_output",
)

In [ ]:
result = run_pipeline(config, connection=connection)
print(result.summary())

## Coastal mode: monitoring land reclamation

The same tool runs in reverse for coasts: water becoming land instead of land replacing water. Set `mode="coastal"` and the report states net land change alongside the water area. Example — Dubai's Palm Jebel Ali, where reclamation restarted in 2023:

```python
config = PipelineConfig(
    aoi=AOI(lat=25.00, lon=55.02, buffer_km=15),   # Palm Jebel Ali / Jebel Ali port
    dates=PipelineConfig.date_range("2019-01-15", "2025-01-15", "yearly"),
    mode="coastal",
    max_cloud_cover=40,
    resolution_m=60,
    cache_dir="/content/drive/MyDrive/aqualens/aqualens_cache",
    out_dir="/content/drive/MyDrive/aqualens/aqualens_output/palm_jebel_ali",
)
result = run_pipeline(config, connection=connection)
print(result.summary())
```

Watch the QC flags: `wide_shoreline_band` means tidal flats are making the shoreline position depend on the tide state of the composited scenes, so treat small changes with caution. (Dubai's tides are small, so this example should be clean.)

In [ ]:
from IPython.display import Image, display

for figure in result.figures:
    display(Image(str(figure)))